# Lecture 4: Proof Hygiene and Boundaries

Proof hygiene is the discipline of checking whether formal artifacts have obvious escape hatches or misleading theorem surfaces. It does not replace proof checking. It catches common ways a proof corpus can look stronger than it is.


## Learning Objectives

- Detect `sorry`, local `axiom`, trivial theorem targets, and suspicious `by trivial`.
- Understand why comments should not be treated as fatal proof failures.
- Explain why manifest coverage matters.
- Distinguish a hygiene warning from a replay failure.
- Write precise boundary language for proof reports.


## Patterns PACTA Scans For

- `sorry`
- `axiom` declarations under `Proofs/`
- theorem targets such as `: True :=`
- suspicious `by trivial` in spec/certificate/root files
- `native_decide` as advisory unless dependency-cone analysis is stronger
- missing certificate names
- proof files not included in a manifest when a manifest exists

A simple scanner may over-warn. It must not over-claim.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from pacta.audit import scan_hygiene
from pacta.manifest import discover_layout

fixture = repo_root / "tests" / "fixtures" / "mini-ed25519-verified"
layout = discover_layout(fixture, "verification")
issues = scan_hygiene(layout, ["CurveFieldProofs.fieldImplementation"])
print("issues:", issues)


## Why `sorry` Is Serious

In Lean, `sorry` can stand in for a proof. Depending on settings, it may allow a theorem to exist without its proof being completed. In a verification-evidence pipeline, unresolved `sorry` must block high-confidence claims.

The lesson is not "never prototype with placeholders." The lesson is "never ship assurance claims that hide placeholders."


## Why Local Axioms Are Serious

A local axiom can assert the result directly. For example:

```lean
axiom fieldImplementation : CorrectFieldImplementation
```

That may be useful for bootstrapping a model, but it is not proof evidence for implementation correctness. PACTA flags local axioms under `Proofs/` because they may collapse the intended theorem into an assumption.


## Trivial Theorems and Spec Drift

A theorem target like `: True := by trivial` proves exactly nothing about cryptographic code. A more subtle failure is spec drift: the theorem proves a property, but not the property the system needs.

Professional review asks two questions:

1. Is the proof complete?
2. Is the theorem the right theorem?


In [ ]:
# A tiny reviewer helper: classify theorem statements by obvious risk.
examples = {
    "good_shape": "theorem add_denote ... : denote (add x y) = x + y := ...",
    "trivial_target": "theorem certificate : True := by trivial",
    "placeholder": "theorem hard_part : P := by sorry",
}

for name, text in examples.items():
    flags = []
    if "sorry" in text:
        flags.append("placeholder proof")
    if ": True :=" in text:
        flags.append("trivial target")
    if "by trivial" in text:
        flags.append("trivial proof tactic")
    print(name, flags or ["needs semantic review"])


## Exercises

- Add a temporary Lean file under a scratch fixture with a theorem `: True := by trivial`. Run the scanner and inspect the issue.
- Explain why the same word in a comment should not be fatal by itself.
- Write a one-page checklist for reviewing a new `*-verified` repository before assigning any risk score above R2.
